In [ ]:
# Cell 1: Clone Heart-Brain Axis AI & install pinned runtime dependencies
import os
import subprocess
import sys

REPO_URL = "https://github.com/George-Web-Developer/heart-brain-ai-cloud.git"
REPO_DIR = "/content/heart-brain-ai-cloud"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

deps = [
    "numpy==2.4.6",
    "scipy==1.17.1",
    "pandas==3.0.6",
    "scikit-learn==1.9.1",
    "xgboost==3.2.0",
    "fastapi==0.143.3",
    "uvicorn==0.54.0",
    "pydantic>=2.6.0",
    "websockets>=12.0",
    "requests",
]
print(f"[Setup] Installing runtime dependencies on Python {sys.version.split()[0]}...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *deps],
    check=True,
)

if not os.path.exists("/usr/local/bin/cloudflared"):
    print("[Setup] Downloading cloudflared...")
    subprocess.run(
        [
            "curl", "-fsSL", "-o", "/usr/local/bin/cloudflared",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ],
        check=True,
    )
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

print("[Setup Complete] Ready to launch FastAPI AI Server + Cloudflare Tunnel in Cell 2.")

In [ ]:
# Cell 2: Start FastAPI AI Server + Public Cloudflare HTTPS Tunnel + Live Logs
import math
import os
import re
import subprocess
import sys
import threading
import time
import requests

REPO_DIR = "/content/heart-brain-ai-cloud"
subprocess.run(["pkill", "-f", "src.server.app:app"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(1)

env = os.environ.copy()
env["CLOUD_MINIMAL_MODELS"] = "1"
env["ENABLE_DEEP_LEARNING"] = "1"
env["OMP_NUM_THREADS"] = "1"
env["PYTHONUNBUFFERED"] = "1"

print("[Server] Starting Heart-Brain Axis AI FastAPI server on 127.0.0.1:8000...")
uvicorn_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.server.app:app", "--host", "127.0.0.1", "--port", "8000", "--workers", "1"],
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def _stream_uvicorn():
    assert uvicorn_proc.stdout is not None
    for line in uvicorn_proc.stdout:
        print("[API]", line.rstrip(), flush=True)

log_thread = threading.Thread(target=_stream_uvicorn, daemon=True)
log_thread.start()

healthy = False
for _ in range(60):
    if uvicorn_proc.poll() is not None:
        raise RuntimeError(f"Uvicorn exited early with code {uvicorn_proc.returncode}")
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print("[Server] Local Health OK:", r.json(), flush=True)
            healthy = True
            break
    except Exception:
        time.sleep(1)

if not healthy:
    raise RuntimeError("Timed out waiting for local FastAPI server /health")

print("[Tunnel] Opening free public Cloudflare HTTPS tunnel...", flush=True)
cf_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
cf_lines = []
deadline = time.time() + 45
assert cf_proc.stdout is not None
while time.time() < deadline:
    line = cf_proc.stdout.readline()
    if not line:
        if cf_proc.poll() is not None:
            break
        time.sleep(0.2)
        continue
    cf_lines.append(line.rstrip())
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError("Failed to acquire trycloudflare.com URL. Tail logs:\n" + "\n".join(cf_lines[-15:]))

def _drain_cf():
    assert cf_proc.stdout is not None
    for _ in cf_proc.stdout:
        pass

threading.Thread(target=_drain_cf, daemon=True).start()

# Wait briefly for public DNS propagation and verify end-to-end inference
ext_ok = False
for _ in range(20):
    try:
        rh = requests.get(f"{public_url}/health", timeout=5)
        if rh.status_code == 200:
            print("[Tunnel] External Health OK:", rh.json(), flush=True)
            ext_ok = True
            break
    except Exception:
        time.sleep(1.5)

if not ext_ok:
    print("[Tunnel Warning] External DNS still propagating; try opening /health in a moment.", flush=True)
else:
    t_vals = [i / 250.0 for i in range(2500)]
    ecg_test = [round(0.05 * math.sin(2.0 * math.pi * 1.2 * t), 4) for t in t_vals]
    ppg_test = [round(62000.0 + 500.0 * math.sin(2.0 * math.pi * 1.2 * t), 1) for t in t_vals]
    rp = requests.post(
        f"{public_url}/api/v1/esp32-predict?compact=true",
        json={"device_id": "COLAB_SELFTEST", "raw_windows": {"ecg_mv": ecg_test, "ppg_ir": ppg_test}},
        timeout=25,
    )
    print("[Self-Test] /api/v1/esp32-predict?compact=true ->", rp.status_code, rp.json().get("summary"), flush=True)

print("\n" + "=" * 72, flush=True)
print(f"PUBLIC_CLOUD_URL: {public_url}", flush=True)
print("=" * 72, flush=True)
print(f'ESP32 cloud_config.h -> #define CLOUD_BASE_URL "{public_url}"', flush=True)
print(f"USB Bridge command   -> python Firmware_ESP32/read_esp32_live.py --cloud-url {public_url}", flush=True)
print("Keep this cell running while your ESP32 / PC streams telemetry.", flush=True)
print("=" * 72 + "\n", flush=True)

try:
    while uvicorn_proc.poll() is None and cf_proc.poll() is None:
        time.sleep(2)
finally:
    uvicorn_proc.terminate()
    cf_proc.terminate()